# 🤖 AI Agents Bootcamp — Module 0 Labs
### From LLMs to Agents: Five Progressive Demonstrations

---

**How to use this notebook:**
- Run cells **one at a time** using `Shift + Enter`
- Read each explanation before running
- Watch the output appear directly below the cell
- Use the **Jupyter Variables** panel (top toolbar) to inspect live data

**What you'll demonstrate:**
| Lab | What It Shows | Key Concept |
|-----|--------------|-------------|
| 0 | API connection + response object | The 5-stage flow |
| 1 | Static LLM call vs Agent loop | The fundamental difference |
| 2 | Agent with tools | Claude deciding what to call |
| 3 | Stateful memory agent | Persistence across turns |
| 4 | Multi-agent orchestration | Supervisor + specialists |

> **API Key:** Your `.env` file must contain `ANTHROPIC_API_KEY=sk-ant-...`  
> **Cost:** All five labs together cost under £0.05 in API usage.

---
## ⚙️ Setup — Run This First, Every Session

In [ ]:
# Run this cell first every time you open the notebook
# It loads your API key and creates the client

import anthropic
import json
from dotenv import load_dotenv

load_dotenv()  # reads ANTHROPIC_API_KEY from your .env file

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"

print("✓ Anthropic client ready")
print(f"  Model: {MODEL}")
print("  Key loaded from .env")

---
## 🔬 Lab 0 — Your First API Call & The Response Object

**What this demonstrates:**  
The complete 5-stage Anthropic API flow. One prompt → one response → done.  
This is a **static LLM call** — not an agent. We start here deliberately so learners can feel the difference when Lab 1 adds the loop.

**While this runs, tell your class:**  
> *"Right now our request is travelling through all 5 stages — Request to Server, to Anthropic API, Model Processing (tokenize → embed → contextualize → generate), Response to Server, Response to us. One round trip."*

In [ ]:
# LAB 0, CELL 1: The simplest possible API call
# This is a STATIC call - one prompt, one response, Claude stops.

response = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[
        {"role": "user", "content": "What is quantum computing? Answer in one sentence."}
    ]
)

# Display the answer
print("Claude's answer:")
print(response.content[0].text)

In [ ]:
# LAB 0, CELL 2: Inspect the full response object
# Show students EXACTLY what the API returns
# Hover over 'response' in Jupyter Variables panel to see live object

print("=" * 50)
print("THE RESPONSE OBJECT")
print("=" * 50)
print(f"  stop_reason:    {response.stop_reason}")
print(f"  input_tokens:   {response.usage.input_tokens}")
print(f"  output_tokens:  {response.usage.output_tokens}")
print(f"  model:          {response.model}")
print()
print("WHAT EACH FIELD MEANS:")
print(f"  stop_reason 'end_turn' = model finished naturally ✓")
print(f"  stop_reason 'max_tokens' = response was cut off ✗")
print(f"  stop_reason 'tool_use' = model wants to call a tool (Lab 2!)")
print()
print(f"  Cost this call: ~${(response.usage.input_tokens * 0.000003 + response.usage.output_tokens * 0.000015):.6f} USD")

In [ ]:
# LAB 0, CELL 3: Multi-turn conversation (like in your screenshot)
# Demonstrate that context is passed manually in the messages array
# Claude has NO memory between calls - YOU maintain the history

print("MULTI-TURN CONVERSATION")
print("Note: we pass the FULL history every time")
print("=" * 50)

# Turn 1
response1 = client.messages.create(
    model=MODEL,
    max_tokens=150,
    messages=[
        {"role": "user", "content": "What is quantum computing? Answer in one sentence."}
    ]
)
answer1 = response1.content[0].text
print(f"Turn 1 — User: What is quantum computing?")
print(f"Turn 1 — Claude: {answer1}")
print()

# Turn 2 — pass full history so Claude remembers Turn 1
response2 = client.messages.create(
    model=MODEL,
    max_tokens=150,
    messages=[
        {"role": "user",      "content": "What is quantum computing? Answer in one sentence."},
        {"role": "assistant", "content": answer1},  # Turn 1 response included
        {"role": "user",      "content": "Write another sentence expanding on that."}  # Turn 2
    ]
)
print(f"Turn 2 — User: Write another sentence expanding on that.")
print(f"Turn 2 — Claude: {response2.content[0].text}")
print()
print("KEY POINT: Without passing Turn 1 history, Claude would not know what 'that' refers to.")
print("This is how ALL agent memory works - you manage the messages array.")

---
## 🔄 Lab 1 — Static Call vs Agent Loop: Spot the Difference

**What this demonstrates:**  
Side-by-side comparison. Static call: Claude answers and stops. Agent loop: Claude reasons, requests a tool, your code runs it, Claude observes the result, continues.

**The question requires two steps:**  
*"What is the temperature in London today, and what is that number multiplied by 5?"*

A static call cannot answer this correctly. An agent can — because it can call `get_weather`, read the result, then call `calculate`.

**Pause before running Cell 2 and ask the class:**  
> *"How many API calls do you think this needs?"* (Answer: 3 turns)

In [ ]:
# LAB 1, CELL 1: Static call attempt — shows the limitation
# Claude will hedge or guess because it has no live weather access

print("STATIC CALL ATTEMPT")
print("=" * 50)
response = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[{
        "role": "user",
        "content": "What is the temperature in London today, and what is that multiplied by 5?"
    }]
)
print(response.content[0].text)
print()
print("⚠ Notice: Claude hedges, guesses, or refuses. It cannot access live data.")
print("   This is Gap 1 from Module 0: No World Access.")

In [ ]:
# LAB 1, CELL 2: Define tools and the agent loop
# Tools are YOUR functions. Claude outputs a tool_use block.
# YOUR code detects it, executes the function, sends the result back.

# ── TOOL FUNCTIONS ────────────────────────────────────────────────
def get_weather(city: str) -> str:
    """Simulated. Replace with real API (OpenWeatherMap is free)."""
    data = {
        "london": "18°C, partly cloudy",
        "new york": "24°C, sunny",
        "tokyo": "29°C, humid",
        "paris": "16°C, overcast",
    }
    return data.get(city.lower(), f"No data for {city}")

def calculate(expression: str) -> str:
    """Safe calculator. Only evaluates math expressions."""
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

# ── TOOL SCHEMAS (what Claude sees) ───────────────────────────────
TOOLS = [
    {
        "name": "get_weather",
        "description": "Get current weather and temperature for a named city. Returns temperature in Celsius.",
        "input_schema": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "City name, e.g. 'London'"}},
            "required": ["city"]
        }
    },
    {
        "name": "calculate",
        "description": "Evaluate a mathematical expression. Example: '18 * 5' returns '90'.",
        "input_schema": {
            "type": "object",
            "properties": {"expression": {"type": "string"}},
            "required": ["expression"]
        }
    }
]

print("✓ Tools defined")
print(f"  get_weather — test: {get_weather('london')}")
print(f"  calculate  — test: {calculate('18 * 5')}")

In [ ]:
# LAB 1, CELL 3: THE AGENT LOOP
# Watch the turn numbers. Claude picks tool order. You execute.
# This is 20 lines. That's all an agent loop is.

def run_agent(question: str, max_turns: int = 10) -> str:
    messages = [{"role": "user", "content": question}]
    print(f"Question: {question}")
    print("=" * 60)

    for turn in range(max_turns):
        print(f"\n[Turn {turn + 1}] Calling Claude...")
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=TOOLS,
            messages=messages
        )
        print(f"  stop_reason: {response.stop_reason}")

        if response.stop_reason == "end_turn":
            for block in response.content:
                if hasattr(block, "text"):
                    print(f"\n✓ Final Answer: {block.text}")
                    return block.text

        elif response.stop_reason == "tool_use":
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    # Claude tells us WHAT to call — we decide HOW
                    if block.name == "get_weather":
                        result = get_weather(**block.input)
                    elif block.name == "calculate":
                        result = calculate(**block.input)
                    else:
                        result = "Unknown tool"
                    print(f"  → Tool call: {block.name}({block.input})")
                    print(f"  ← Result:    {result}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })
            messages.append({"role": "user", "content": tool_results})

    return "Max turns reached."


# RUN IT — watch Claude pick the tool order
answer = run_agent(
    "What is the temperature in London today, and what is that number multiplied by 5?"
)

### 💬 Debrief — What Just Happened?

- **Turn 1:** Claude reasoned it needs weather first → `stop_reason = 'tool_use'`
- **Turn 2:** Claude saw the temperature → used it to build the calculator input → `stop_reason = 'tool_use'`  
- **Turn 3:** Claude had all the data → `stop_reason = 'end_turn'`

**You never told Claude which tool to call first.** It decided based on what it needed.  
**The loop is the agent.** The model is the same Claude you called in Lab 0. The architecture changed, not the model.

---
## 🧠 Lab 2 — Stateful Agent: Memory Across Turns

**What this demonstrates:**  
Gap 2 solved: No Persistence. The agent can save facts to an external memory store, recall them when asked, and update them when data changes.

**Watch for:** `[MEMORY SAVED]` and `[MEMORY RECALL]` tags in the output.  
These show you exactly when Claude decides to use the memory tools — you never told it to.

**Ask the class before running:**  
> *"If we close this notebook and reopen it, will it remember Project Alpha?"*  
> *(Answer: No — the dict is in-memory. In production you'd use a database.)*

In [ ]:
# LAB 2, CELL 1: Memory store and tools

# External memory — survives across turns in this session
# In production: replace with Redis, DynamoDB, or a vector store
memory_store = {}

def remember(key: str, value: str) -> str:
    memory_store[key] = value
    print(f"  [MEMORY SAVED] {key} = {value}")
    return f"Saved: {key} = {value}"

def recall(key: str) -> str:
    value = memory_store.get(key, f"Nothing stored for '{key}'")
    print(f"  [MEMORY RECALL] {key} → {value}")
    return str(value)

def list_all_memories() -> str:
    if not memory_store:
        return "Memory is empty."
    return "\n".join([f"{k}: {v}" for k, v in memory_store.items()])

MEMORY_TOOLS = [
    {"name": "remember",
     "description": "Save a fact to memory. Use this whenever the user shares important information.",
     "input_schema": {"type": "object",
         "properties": {"key": {"type": "string"}, "value": {"type": "string"}},
         "required": ["key", "value"]}},
    {"name": "recall",
     "description": "Look up a stored fact by key. Use this before answering questions about past information.",
     "input_schema": {"type": "object",
         "properties": {"key": {"type": "string"}},
         "required": ["key"]}},
    {"name": "list_all_memories",
     "description": "List everything in memory.",
     "input_schema": {"type": "object", "properties": {}}}
]

conversation = []  # shared across all turns — this is short-term memory

print("✓ Memory tools ready")
print("  memory_store:", memory_store)

In [ ]:
# LAB 2, CELL 2: The memory agent

def memory_agent_turn(user_input: str) -> str:
    """Single turn of the stateful agent. Appends to shared conversation."""
    print(f"\n── You: {user_input}")
    conversation.append({"role": "user", "content": user_input})

    for _ in range(8):
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            system="You are a helpful project assistant. Always use 'remember' to save important info the user shares. Always use 'recall' before answering questions about anything they may have told you.",
            tools=MEMORY_TOOLS,
            messages=conversation
        )

        if response.stop_reason == "end_turn":
            answer = next((b.text for b in response.content if hasattr(b, "text")), "")
            conversation.append({"role": "assistant", "content": response.content})
            print(f"── Agent: {answer}")
            return answer

        conversation.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                if block.name == "remember":        result = remember(**block.input)
                elif block.name == "recall":         result = recall(**block.input)
                elif block.name == "list_all_memories": result = list_all_memories()
                else:                                result = "Unknown tool"
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
        conversation.append({"role": "user", "content": tool_results})

    return "Max turns"

print("✓ memory_agent_turn() ready — run Turn 1 below")

In [ ]:
# LAB 2, CELL 3: Turn 1 — give the agent information
memory_agent_turn("Project Alpha has a budget of £50,000 and is due in December.")

In [ ]:
# LAB 2, CELL 4: Turn 2 — ask it to recall
# Notice it uses 'recall' before answering — you didn't tell it to
memory_agent_turn("What is the budget for Project Alpha?")

In [ ]:
# LAB 2, CELL 5: Turn 3 — update the data
memory_agent_turn("The board approved an increase — Project Alpha budget is now £75,000.")

In [ ]:
# LAB 2, CELL 6: Turn 4 — full summary (did it remember the update?)
memory_agent_turn("Give me the full current status of Project Alpha.")

In [ ]:
# LAB 2, CELL 7: Inspect what's in memory — open the Variables panel too
print("Current memory_store contents:")
for k, v in memory_store.items():
    print(f"  {k}: {v}")

print(f"\nConversation turns so far: {len(conversation)}")
print("(Each turn = one message in the messages array Claude received)")

---
## 🔗 Lab 3 — Tool Chaining: Agent Builds Its Own Workflow

**What this demonstrates:**  
Three tools. No hardcoded workflow. Claude decides which to call, in what order, based on what it discovers.  
This is Anthropic's **Autonomous Agent** pattern — the LLM dynamically directs its own processes.

**Watch the turn sequence.** Claude will:
1. Search first (needs context)
2. Read specific results (needs detail)
3. Save a note (task complete)

You gave it none of those instructions — it built the workflow from the task description alone.

In [ ]:
# LAB 3, CELL 1: Define 3 tools and the research agent

notes_db = {}  # where the agent saves its research

def web_search(query: str) -> str:
    print(f"  [SEARCH] {query}")
    # Simulated results — replace with Brave Search API or SerpAPI for live data
    results = {
        "ai agents": """Results for: ai agents
1. Klarna AI agent: 2.3M conversations/month, equivalent to 700 FTEs, $40M saving
2. GitHub Copilot Workspace: autonomous bug fixing from issue descriptions
3. Anthropic 'Building Effective Agents' guide — Dec 2024: 5 patterns defined
4. Harvey AI at Allen & Overy: contract review 40 hours → 4 minutes
5. AWS DevOps Agent: resolves tier-1 incidents without human intervention""",
        "agentic patterns": """Results: Anthropic Dec 2024 defines 5 patterns:
1. Prompt Chaining — sequential steps with gates
2. Routing — classify then direct to specialist
3. Parallelisation — independent subtasks in parallel
4. Orchestrator-Workers — dynamic task delegation
5. Evaluator-Optimizer — generate then critique in a loop"""
    }
    for key in results:
        if any(w in query.lower() for w in key.split()):
            return results[key]
    return f"General results for '{query}': Agentic AI is growing rapidly across industries."

def read_source(topic: str) -> str:
    print(f"  [READ] {topic}")
    sources = {
        "klarna": "Klarna AI agent: 2.3M conversations/month. Same CSAT as human agents. $40M annual saving. Publicly disclosed in Klarna earnings 2024. Handles tier-1 and tier-2 queries end-to-end.",
        "anthropic": "Anthropic's guide (Dec 2024): Start simple. Only add complexity when needed. Three principles: simplicity, transparency, good tool design. Agents = LLMs that dynamically direct their own processes.",
        "github": "GitHub Copilot Workspace: reads GitHub issue → plans file changes → writes code → runs tests → opens PR. 30-55% developer productivity increase in controlled studies.",
        "production": "Production agents need: guardrails (max turns, cost budget), observability (trace every decision), retry logic, graceful failure modes. Treat agents like distributed systems."
    }
    for k, v in sources.items():
        if k in topic.lower():
            return v
    return f"Content on '{topic}': Significant impact across industries."

def save_note(title: str, content: str) -> str:
    notes_db[title] = content
    print(f"  [SAVED] '{title}' ({len(content)} chars)")
    return f"Note saved: '{title}'"

RESEARCH_TOOLS = [
    {"name": "web_search", "description": "Search for recent information on a topic.",
     "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}},
    {"name": "read_source", "description": "Get detailed info on a topic: klarna, anthropic, github, or production.",
     "input_schema": {"type": "object", "properties": {"topic": {"type": "string"}}, "required": ["topic"]}},
    {"name": "save_note", "description": "Save a research note with title and content.",
     "input_schema": {"type": "object",
         "properties": {"title": {"type": "string"}, "content": {"type": "string"}},
         "required": ["title", "content"]}}
]

print("✓ Research tools ready")

In [ ]:
# LAB 3, CELL 2: Run the research agent
# Watch it build: search → read → save. You never specified this order.

task = "Research AI agents in 2024 — what patterns exist, what's in production, and save a comprehensive summary note."
print(f"Task: {task}")
print("=" * 60)

messages = [{"role": "user", "content": task}]

for turn in range(15):
    response = client.messages.create(
        model=MODEL,
        max_tokens=2048,
        system="You are a research agent. For research tasks: search first, then read relevant sources, then save a comprehensive summary note. Be thorough.",
        tools=RESEARCH_TOOLS,
        messages=messages
    )
    print(f"\n[Turn {turn+1}] {response.stop_reason}")

    if response.stop_reason == "end_turn":
        answer = next((b.text for b in response.content if hasattr(b, "text")), "")
        print(f"\n✓ Done. Notes saved: {list(notes_db.keys())}")
        print(f"\nAgent summary: {answer}")
        break

    messages.append({"role": "assistant", "content": response.content})
    tool_results = []
    for block in response.content:
        if block.type == "tool_use":
            if block.name == "web_search":   result = web_search(**block.input)
            elif block.name == "read_source": result = read_source(**block.input)
            elif block.name == "save_note":   result = save_note(**block.input)
            else:                             result = "Unknown"
            tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
    messages.append({"role": "user", "content": tool_results})

In [ ]:
# LAB 3, CELL 3: Read the saved note
print("Saved notes:")
for title, content in notes_db.items():
    print(f"\n── {title}")
    print(content)

---
## 🏢 Lab 4 — Multi-Agent: Supervisor + Specialist Agents

**What this demonstrates:**  
Anthropic's **Orchestrator-Workers** pattern. Three separate Claude API calls.  
Each agent has a different system prompt, different tools, different responsibility.

```
SUPERVISOR
  ├── delegates to → RESEARCH AGENT (has search + read tools)
  └── delegates to → WRITING AGENT  (no tools, just writes)
         └── supervisor reviews + delivers
```

**Watch for:** Each agent prints its own header so you can see which is active.  
**Key insight:** The supervisor never does the actual work — it only coordinates.

In [ ]:
# LAB 4, CELL 1: Research Specialist Agent

def research_agent(task: str) -> str:
    """Specialist: gathers facts using search tools. Returns structured data."""
    print("\n  ══ RESEARCH AGENT activated ══")

    # Reuse the tools from Lab 3
    messages = [{"role": "user", "content": task}]

    for _ in range(10):
        response = client.messages.create(
            model=MODEL,
            max_tokens=2048,
            system="You are a specialist research agent. Use your tools to gather comprehensive, factual information with specific numbers, named examples, and key insights. Return a structured summary.",
            tools=RESEARCH_TOOLS,  # same tools from Lab 3
            messages=messages
        )
        if response.stop_reason == "end_turn":
            result = next((b.text for b in response.content if hasattr(b, "text")), "")
            print(f"  ══ RESEARCH AGENT done ({len(result)} chars) ══")
            return result

        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                if block.name == "web_search":   r = web_search(**block.input)
                elif block.name == "read_source": r = read_source(**block.input)
                elif block.name == "save_note":   r = save_note(**block.input)
                else:                             r = "Unknown"
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": r})
        messages.append({"role": "user", "content": tool_results})
    return "Research incomplete"

print("✓ research_agent() ready")

In [ ]:
# LAB 4, CELL 2: Writing Specialist Agent

def writing_agent(topic: str, research_data: str) -> str:
    """Specialist: turns research into a polished briefing. No tools needed."""
    print("\n  ══ WRITING AGENT activated ══")

    response = client.messages.create(
        model=MODEL,
        max_tokens=2048,
        system="You are a specialist writing agent. Transform research data into clear, professional briefings. Use headers. Include specific statistics. Write in plain English. No jargon.",
        messages=[{
            "role": "user",
            "content": f"Write a professional briefing on: {topic}\n\nResearch data:\n{research_data}\n\nFormat: Executive Summary (2-3 sentences), Key Facts (5 bullets with numbers), What It Means For Teams (3 practical points), Bottom Line (1 sentence)."
        }]
    )
    result = response.content[0].text
    print(f"  ══ WRITING AGENT done ({len(result)} chars) ══")
    return result

print("✓ writing_agent() ready")

In [ ]:
# LAB 4, CELL 3: The Supervisor — coordinates without doing the work itself

def supervisor(user_request: str) -> str:
    """Orchestrator: reads task, delegates to specialists, reviews, delivers."""
    print(f"\n{'='*60}")
    print("SUPERVISOR: Reading task and planning delegation...")
    print(f"Task: {user_request}")
    print(f"{'='*60}")

    # Step 1: Delegate to Research Agent
    print("\nSUPERVISOR → Research Agent: gather data")
    research_data = research_agent(f"Research comprehensively: {user_request}")

    # Step 2: Delegate to Writing Agent with the research
    print("\nSUPERVISOR → Writing Agent: produce briefing")
    report = writing_agent(user_request, research_data)

    # Step 3: Supervisor reviews (light quality check + delivery)
    print("\nSUPERVISOR: Reviewing and delivering...")
    final = client.messages.create(
        model=MODEL,
        max_tokens=100,
        system="You are a supervisor reviewing a briefing. Output ONLY one line: 'QUALITY: [brief assessment]' — nothing else.",
        messages=[{"role": "user", "content": f"Review this briefing:\n{report[:500]}..."}]
    ).content[0].text

    print(f"\nSupervisor review: {final}")
    return report

print("✓ supervisor() ready — run the demo below")

In [ ]:
# LAB 4, CELL 4: Run the full multi-agent system
# Count the agents: 1 supervisor + 1 research + 1 writing = 3 Claude API calls

final_report = supervisor(
    "The state of AI agents in 2024 and what it means for technology teams."
)

print(f"\n{'='*60}")
print("FINAL DELIVERED REPORT:")
print(f"{'='*60}")
print(final_report)

### 💬 Debrief — What Just Happened?

**Three separate Claude calls.** Each with a different role:
- **Research Agent:** Specialised in gathering facts. Has tools.
- **Writing Agent:** Specialised in clear prose. No tools — it only writes.
- **Supervisor:** Reads the brief, delegates, quality-checks. Never does the actual work.

**Why separate?** Each agent has a focused system prompt. Focused prompts = better outputs than one agent doing everything.

**In production** (from Anthropic's guidance):
> *"Use orchestrator-workers when you can't predict the subtasks needed upfront and flexibility matters more than predictability."*

---
## ✅ Summary — What You've Demonstrated

| Lab | You ran | The concept it proved |
|-----|---------|----------------------|
| 0 | `client.messages.create()` | The 5-stage API flow. The response object. Multi-turn context. |
| 1 | Static call vs agent loop | Static LLM hits a ceiling. The loop is what makes agents work. |
| 2 | Memory agent | Persistence. External state store. `remember()` + `recall()`. |
| 3 | Research agent | Tool chaining. Claude builds its own workflow. No hardcoded order. |
| 4 | Multi-agent system | Orchestrator-Workers pattern. Supervisor + specialists. 3 Claude calls. |

### What's the same across all 5 labs?
`client.messages.create()` — one function call through the same 5-stage flow.

What changed: the messages array, the tools, and whether there's a loop.

---
*AI Agents Bootcamp · Module 0 Labs · Run with VS Code Jupyter Extension + Anthropic SDK*